Практика по PyTorch

Цель:
получить навыки практической работы с PyTorch.


Описание/Пошаговая инструкция выполнения домашнего задания:
Используя PyTorch:

Создать и обучить регрессионную модель, которая аппроксимирует значение функции sin(x + 2*y)exp(-(2x + y)^2) на диапазоне значений [-10;10] по х и у.
В качестве обучающей выборки необходимо сгенерировать 20000 точек случайным образом.
Разделить получившийся датасет на train / test / val в отношениях 70% / 15% / 15%, соответственно.

Результаты:

a) Посчитать метрику Mean Square Error(MSE) на test.

b) Нарисовать график, в котором сравнивается истинная функция и ее аппроксимированный вариант (предсказываемый моделью).


Критерии оценки:
"Принято" - задание выполнено полностью.

In [35]:
# =====================
# 1. Setup configs
# =====================

import os
import time
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

# Configs
RANDOM_STATE = 555
pd.set_option('display.max_columns', None)

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Directories
output_dir = 'csv_output'
os.makedirs(output_dir, exist_ok=True)

print('PyTorch version:', torch.__version__)
print('Device:', device)

PyTorch version: 2.12.1+cpu
Device: cpu


In [36]:
# --------------------
# 2. Generate the dataset
# --------------------

N_POINTS = 20000
COORD_RANGE = (-10.0, 10.0)


def target_func(x, y):
    # f(x, y) = sin(x + 2y) * exp(-(2x + y)^2)
    return np.sin(x + 2 * y) * np.exp(-(2 * x + y) ** 2)


# Random numpy generator for 20000 points [-10, 10]
rng = np.random.default_rng(RANDOM_STATE)
xy = rng.uniform(COORD_RANGE[0], COORD_RANGE[1], size=(N_POINTS, 2)).astype(np.float32)
z = target_func(xy[:, 0], xy[:, 1]).astype(np.float32)

df_raw = pd.DataFrame({'x': xy[:, 0], 'y': xy[:, 1], 'z': z})
display(df_raw.head())

print('Dataset shape:', df_raw.shape)
print('Target function z: min = {:.6f}, max = {:.6f}, mean = {:.6f}, std = {:.6f}'.format(
    z.min(), z.max(), z.mean(), z.std()))
print('Share of points with |z| < 0.001: {:.3%}'.format((np.abs(z) < 0.001).mean()))

,x,y,z
0,-5.684924,-9.751722,-0.000000e+00
1,-9.475870,4.410860,-0.000000e+00
2,-8.463186,7.822143,7.867836e-37
3,-6.054430,3.339083,2.319067e-34
4,7.770640,1.598879,-0.000000e+00


Dataset shape: (20000, 3)
Target function z: min = -0.999676, max = 0.999641, mean = -0.000444, std = 0.125297
Share of points with |z| < 0.001: 87.470%


In [37]:
# --------------------
# 3. Target actual function
# --------------------

# The gaussian factor exp(-(2x + y)^2) keeps the signal only inside a narrow
# diagonal strip, everywhere else the function is almost zero.
# That strip is exactly what the network has to learn from random points.

grid_pts = np.linspace(COORD_RANGE[0], COORD_RANGE[1], 100)
gx, gy = np.meshgrid(grid_pts, grid_pts)
gz = target_func(gx, gy)

fig = go.Figure(data=go.Surface(
    x=grid_pts, y=grid_pts, z=gz,
    colorscale='RdBu', cmid=0, colorbar_title='f(x, y)'))

fig.update_layout(
    title='True function: sin(x + 2y) * exp(-(2x + y)^2)',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='f(x, y)', bgcolor='rgb(58, 58, 58)'),
    width=800, height=600)

fig.show()

In [38]:
# --------------------
# 4. Train / test / val split 70/15/15
# --------------------

X_train, X_temp, z_train_arr, z_temp = train_test_split(
    xy, z, test_size=0.30, random_state=RANDOM_STATE)
X_test, X_val, z_test_arr, z_val_arr = train_test_split(
    X_temp, z_temp, test_size=0.50, random_state=RANDOM_STATE)

print('Train points:', len(X_train), '(70%)')
print('Test points: ', len(X_test), '(15%)')
print('Val points:  ', len(X_val), '(15%)')

# input data statistics
x_mean = X_train.mean(axis=0)
x_std = X_train.std(axis=0)


def to_tensor(X_arr, z_arr):
    X_scaled = (X_arr - x_mean) / x_std
    return (torch.tensor(X_scaled, dtype=torch.float32),
            torch.tensor(z_arr, dtype=torch.float32).unsqueeze(1))


X_train_t, z_train_t = to_tensor(X_train, z_train_arr)
X_test_t, z_test_t = to_tensor(X_test, z_test_arr)
X_val_t, z_val_t = to_tensor(X_val, z_val_arr)

BATCH_SIZE = 256
loader_g = torch.Generator().manual_seed(RANDOM_STATE)
train_loader = DataLoader(
    TensorDataset(X_train_t, z_train_t),
    batch_size=BATCH_SIZE, shuffle=True, generator=loader_g)

print('Batches per epoch:', len(train_loader))

Train points: 14000 (70%)
Test points:  3000 (15%)
Val points:   3000 (15%)
Batches per epoch: 55


In [39]:
# --------------------
# 5. Multi layer model and training
# --------------------

class RegressionMLP(nn.Module):
    def __init__(self, n_inputs=2, hidden_size=512, n_hidden=4):
        # fast run config: hidden_size=256, n_hidden=3
        super().__init__()
        layers = [nn.Linear(n_inputs, hidden_size), nn.GELU()]
        # nn.RELU() demostrates higher error, so we use GELU
        for _ in range(n_hidden - 1):
            layers += [nn.Linear(hidden_size, hidden_size), nn.GELU()]
        layers += [nn.Linear(hidden_size, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x_in):
        return self.net(x_in)


model = RegressionMLP().to(device)
print(model)
print('Number of parameters:', sum(p.numel() for p in model.parameters()))

EPOCHS = 500
LR = 0.001 # lerning rate 1e-3 as size of offset for next train run

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
loss_fn = nn.MSELoss()

history = {'train': [], 'val': []}
best_val = float('inf')
best_state = None

t_start = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for xb, zb in train_loader:
        xb, zb = xb.to(device), zb.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(xb), zb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * xb.size(0)
    scheduler.step()

    model.eval()
    with torch.no_grad():
        val_loss = loss_fn(model(X_val_t.to(device)), z_val_t.to(device)).item()
    history['train'].append(epoch_loss / len(X_train_t))
    history['val'].append(val_loss)

    if val_loss < best_val:
        best_val = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    if epoch == 1 or epoch % 50 == 0:
        print('Epoch {:3d} | train MSE {:.7f} | val MSE {:.7f}'.format(
            epoch, history['train'][-1], val_loss))

train_time = time.time() - t_start
print('\nTraining time: {:.1f} s'.format(train_time))

# Restore the best weights by validation MSE
model.load_state_dict(best_state)
model.eval()
print('Best val MSE: {:.7f}'.format(best_val))

RegressionMLP(
  (net): Sequential(
    (0): Linear(in_features=2, out_features=512, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): GELU(approximate='none')
    (4): Linear(in_features=512, out_features=512, bias=True)
    (5): GELU(approximate='none')
    (6): Linear(in_features=512, out_features=512, bias=True)
    (7): GELU(approximate='none')
    (8): Linear(in_features=512, out_features=1, bias=True)
  )
)
Number of parameters: 790017
Epoch   1 | train MSE 0.0162367 | val MSE 0.0147634
Epoch  50 | train MSE 0.0002370 | val MSE 0.0002871
Epoch 100 | train MSE 0.0000902 | val MSE 0.0001442
Epoch 150 | train MSE 0.0000470 | val MSE 0.0001736
Epoch 200 | train MSE 0.0000477 | val MSE 0.0000478
Epoch 250 | train MSE 0.0000502 | val MSE 0.0000603
Epoch 300 | train MSE 0.0000088 | val MSE 0.0000037
Epoch 350 | train MSE 0.0000018 | val MSE 0.0000017
Epoch 400 | train MSE 0.0000012 | val MSE 0.0000010
Epoch 450 | train M

In [40]:
# --------------------
# 6. Train curve
# --------------------

fig = go.Figure()
fig.add_trace(go.Scatter(
    y=history['train'], name='train MSE',
    line=dict(color='steelblue')))
fig.add_trace(go.Scatter(
    y=history['val'], name='val MSE',
    line=dict(color='firebrick')))
fig.update_layout(
    title='Training progress',
    paper_bgcolor='#3a3a3a', plot_bgcolor='#3a3a3a',
    xaxis_title='epoch', yaxis_title='MSE (log scale)',
    yaxis_type='log', width=800, height=500)
fig.show()

In [41]:
# --------------------
# 7. Test evaluation
# --------------------

with torch.no_grad():
    pred_test = model(X_test_t.to(device)).cpu().numpy().ravel()

mse_test = float(np.mean((pred_test - z_test_arr) ** 2))
baseline_mse = float(np.mean((z_train_arr.mean() - z_test_arr) ** 2))

print('Baseline (predict train mean) MSE on test: {:.7f}'.format(baseline_mse))
print('MSE on test: {:.7f}'.format(mse_test))
print('RMSE on test: {:.7f}'.format(np.sqrt(mse_test)))
print('Improvement over baseline: {:.0f} times'.format(baseline_mse / mse_test))

# Log the result
results_path = os.path.join(output_dir, 'model_results.csv')
new_row = pd.DataFrame([{
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'model': 'RegressionMLP_TEST',
    'hyperparameters': {'hidden_size': 512, 'n_hidden': 4, 'activation': 'GELU',
                        'optimizer': 'Adam', 'lr': LR, 'batch_size': BATCH_SIZE,
                        'epochs': EPOCHS, 'input_scaling': 'standardized on train'},
    'mse_test': round(mse_test, 7),
    'mse_val': round(best_val, 7),
    'baseline_mse_test': round(baseline_mse, 7)}])
if os.path.exists(results_path):
    results_df = pd.concat([pd.read_csv(results_path), new_row], ignore_index=True)
else:
    results_df = new_row
results_df.to_csv(results_path, index=False, float_format='%.7f')

display(results_df.drop_duplicates(subset='model', keep='last').reset_index(drop=True))

Baseline (predict train mean) MSE on test: 0.0174389
MSE on test: 0.0000004
RMSE on test: 0.0006575
Improvement over baseline: 40337 times


,timestamp,model,hyperparameters,mse_test,mse_val,baseline_mse_test
0,2026-11-15 22:20:52,RegressionMLP_TEST,"{'hidden_size': 512, 'n_hidden': 4, 'activatio...",4.000000e-07,4.000000e-07,0.017439


In [42]:
# --------------------
# 8. Actual given function vs model prediction
# --------------------

# Model prediction on the same dense grid as the actual function
grid_scaled = torch.tensor(
    (np.stack([gx.ravel(), gy.ravel()], axis=1) - x_mean) / x_std,
    dtype=torch.float32)
with torch.no_grad():
    pred_grid = model(grid_scaled.to(device)).cpu().numpy().reshape(gz.shape)

err_grid = np.abs(gz - pred_grid)

# Shared color limits for true and predicted surfaces
z_min, z_max = -1.0, 1.0

fig = make_subplots(
    rows=1, cols=3, horizontal_spacing=0.08,
    specs=[[{'type': 'surface'}] * 3],
    subplot_titles=('True function', 'Model prediction', 'Absolute error'))

fig.add_trace(go.Surface(
    x=grid_pts, y=grid_pts, z=gz, colorscale='RdBu', cmin=z_min, cmax=z_max,
    colorbar=dict(x=0.26, len=0.85, title='f(x, y)')), row=1, col=1)
fig.add_trace(go.Surface(
    x=grid_pts, y=grid_pts, z=pred_grid, colorscale='RdBu', cmin=z_min, cmax=z_max, showscale=False), row=1, col=2)
fig.add_trace(go.Surface(
    x=grid_pts, y=grid_pts, z=err_grid, colorscale='Viridis',
    colorbar=dict(x=0.98, len=0.85, title='|error|')), row=1, col=3)

scene_axes = dict(xaxis_title='x', yaxis_title='y', zaxis_title='f(x, y)', bgcolor='rgb(58, 58, 58)')

fig.update_layout(
    title='True function vs model approximation (shared color scale) and absolute error',
    width=1500, height=500,
    scene=dict(**scene_axes), scene2=dict(**scene_axes), scene3=dict(**scene_axes))

fig.show()

print('Grid max absolute error: {:.4f}'.format(err_grid.max()))

Grid max absolute error: 0.0223


In [43]:
# True vs predicted on the test part
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=z_test_arr, y=pred_test, mode='markers', name='test points', opacity=0.5,
    marker=dict(size=4, color='steelblue')))
lims = [-1.05, 1.05]
fig.add_trace(go.Scatter(
    x=lims, y=lims, mode='lines', name='y = x',
    line=dict(color='firebrick', dash='dash')))
fig.update_layout(
    title='Predicted vs true values on test',
    paper_bgcolor='#3a3a3a', plot_bgcolor='#3a3a3a',
    xaxis_title='true f(x, y)', yaxis_title='predicted f(x, y)',
    width=800, height=600)
fig.update_xaxes(range=lims)
fig.update_yaxes(range=lims)
fig.show()

In [47]:
# =====================
# 9. Conclusion
# =====================
# Общее число 20000 точек по заданию разбили на train / test / val = 70% 15% / 15% (14000 / 3000 / 3000).
# Сама заданная функция:
# за счет exp(-(2x + y)^2 получается диагональная полоса значений, все остальные значения стремятся к нулю.

# Для сравнения попробал ReLU и GELU активаторы, гаусовский вариант показал меньшие ошибки.
# шаг изменения весов - 0.001, 500 эпох. оптимизатор Адам.
#
# итоговые реультаты в csv в output папке:
# mse_test,mse_val,
# 0.0000073,0.0000059 (ReLU)
# 0.0000004,0.0000004 (GELU)
# 0.0174389 (train mean baseline)
#
# При прочих равных гаусовский метод больше чем 15раз лучше сработал чем линейный relu
# разница между треин и тест у гаусовского метода сдвигается на 2знака после запятой, поэтому не видна на -7 (может будет видна если добавить разрядность в значениях)

# Сравнивая итог по ошибке GELU (0.0000004) со средним базовым значением (0.0174389),
# получается что применение GELU дает лучшую точность в десятки тысяч раз, как-то подозрительно большая разница

# В целом получается, что настройки по архитектуре модели могут дать значительный прирост к точности по отношению к базовому значению ошибки.
# В конкретном случае  само по себе использование гаусовского способа вместо линейного (при примерно одинаковом времени чуть меньше 5 мин) дает огромный прирост по точности.

